# BRZ — Ingest Raw Logs to Bronze Layer

**Purpose:** Stream raw log events from Azure Event Hubs into the Bronze ADLS Gen2 container as Parquet,  
partitioned by `ingestion_date` and `source_system`.  
**Sources:** `transaction_logs`, `auth_logs`, `api_gateway_logs` Event Hub consumer groups  
**Destination:** `abfss://bronze@<storage_account>.dfs.core.windows.net/logs/`  
**Schedule:** Triggered by ADF / Synapse Pipeline (streaming micro-batch, trigger interval 5 min)

## 1. Parameters

In [ ]:
# Pipeline-injected parameters
batch_id            = "manual_run"          # ADF pipeline run ID — overridden at runtime
storage_account     = "adlsgen2banking"     # ADLS Gen2 storage account name
key_vault_name      = "kv-banking-ai"       # Azure Key Vault name
bronze_container    = "bronze"              # Target ADLS container
checkpoint_base     = "abfss://bronze@adlsgen2banking.dfs.core.windows.net/_checkpoints"
trigger_interval_s  = 300                   # Streaming trigger interval in seconds
max_files_per_trigger = 1000               # Back-pressure control

## 2. Imports and Spark Session

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, LongType, TimestampType,
    DoubleType, IntegerType, BinaryType
)
import datetime
import json

spark = SparkSession.builder.getOrCreate()
spark.conf.set("spark.sql.shuffle.partitions", "200")
spark.conf.set("spark.databricks.delta.optimizeWrite.enabled", "true")

print(f"Spark version  : {spark.version}")
print(f"Batch ID       : {batch_id}")
print(f"Storage Account: {storage_account}")

## 3. Retrieve Event Hub Connection Strings from Key Vault

In [ ]:
from notebookutils import mssparkutils

def get_secret(secret_name: str) -> str:
    """Retrieve a secret from Azure Key Vault via mssparkutils."""
    return mssparkutils.credentials.getSecret(key_vault_name, secret_name)

# Event Hub connection strings (stored as secrets in Key Vault)
eh_conn_txn     = get_secret("eh-connection-transaction-logs")
eh_conn_auth    = get_secret("eh-connection-auth-logs")
eh_conn_apigw   = get_secret("eh-connection-api-gateway-logs")

print("Key Vault secrets retrieved successfully.")

## 4. ADLS Gen2 Storage Configuration

In [ ]:
# Retrieve storage account key from Key Vault and configure Spark
storage_account_key = get_secret("adls-storage-account-key")

spark.conf.set(
    f"fs.azure.account.key.{storage_account}.dfs.core.windows.net",
    storage_account_key
)

def bronze_path(source_system: str) -> str:
    return f"abfss://{bronze_container}@{storage_account}.dfs.core.windows.net/logs/{source_system}"

def checkpoint_path(source_system: str) -> str:
    return f"{checkpoint_base}/{source_system}"

print("ADLS Gen2 storage configuration applied.")

## 5. Event Hub Connection Helpers

In [ ]:
def build_eventhubs_conf(connection_string: str, consumer_group: str = "$Default") -> dict:
    """
    Build the configuration dictionary for the Azure Event Hubs Spark connector.
    Uses the azure-eventhubs-spark library available on Synapse runtimes.
    """
    return {
        "eventhubs.connectionString": spark._jvm.org.apache.spark.eventhubs.EventHubsUtils.encrypt(
            connection_string
        ),
        "eventhubs.consumerGroup": consumer_group,
        # Start from the latest offset on first run; checkpointing handles restarts
        "eventhubs.startingPosition": json.dumps({"offset": "-1", "seqNo": -1, "enqueuedTime": None, "isInclusive": True}),
        "eventhubs.maxEventsPerTrigger": str(max_files_per_trigger),
    }

print("Event Hubs configuration helper defined.")

## 6. Common Enrichment Logic

In [ ]:
def enrich_stream(df, source_system: str):
    """
    Decode the Event Hub binary body, add lineage / audit columns, and
    derive the ingestion_date partition column.
    """
    ingestion_ts = F.current_timestamp()

    return (
        df
        # Event Hub body is binary — cast to string (UTF-8 JSON payload)
        .withColumn("raw_payload",    F.col("body").cast(StringType()))
        .withColumn("source_system",  F.lit(source_system))
        .withColumn("batch_id",       F.lit(batch_id))
        .withColumn("ingestion_timestamp", ingestion_ts)
        .withColumn("ingestion_date",  F.to_date(ingestion_ts))   # partition column
        # Retain Event Hub system properties for replayability
        .withColumn("eh_offset",           F.col("offset").cast(LongType()))
        .withColumn("eh_sequence_number",  F.col("sequenceNumber").cast(LongType()))
        .withColumn("eh_enqueued_time",    F.col("enqueuedTime").cast(TimestampType()))
        .withColumn("eh_partition_id",     F.col("partition").cast(StringType()))
        .select(
            "raw_payload",
            "source_system",
            "batch_id",
            "ingestion_timestamp",
            "ingestion_date",
            "eh_offset",
            "eh_sequence_number",
            "eh_enqueued_time",
            "eh_partition_id",
        )
    )

print("Enrichment function defined.")

## 7. Stream: transaction_logs

In [ ]:
SOURCE_TXN = "transaction_logs"

raw_txn_stream = (
    spark.readStream
    .format("eventhubs")
    .options(**build_eventhubs_conf(eh_conn_txn, consumer_group="cg-bronze-txn"))
    .load()
)

enriched_txn = enrich_stream(raw_txn_stream, SOURCE_TXN)

query_txn = (
    enriched_txn.writeStream
    .format("parquet")
    .option("checkpointLocation", checkpoint_path(SOURCE_TXN))
    .option("path", bronze_path(SOURCE_TXN))
    .partitionBy("ingestion_date", "source_system")
    .trigger(processingTime=f"{trigger_interval_s} seconds")
    .outputMode("append")
    .queryName(f"brz_{SOURCE_TXN}")
    .start()
)

print(f"Streaming query '{query_txn.name}' started — ID: {query_txn.id}")

## 8. Stream: auth_logs

In [ ]:
SOURCE_AUTH = "auth_logs"

raw_auth_stream = (
    spark.readStream
    .format("eventhubs")
    .options(**build_eventhubs_conf(eh_conn_auth, consumer_group="cg-bronze-auth"))
    .load()
)

enriched_auth = enrich_stream(raw_auth_stream, SOURCE_AUTH)

query_auth = (
    enriched_auth.writeStream
    .format("parquet")
    .option("checkpointLocation", checkpoint_path(SOURCE_AUTH))
    .option("path", bronze_path(SOURCE_AUTH))
    .partitionBy("ingestion_date", "source_system")
    .trigger(processingTime=f"{trigger_interval_s} seconds")
    .outputMode("append")
    .queryName(f"brz_{SOURCE_AUTH}")
    .start()
)

print(f"Streaming query '{query_auth.name}' started — ID: {query_auth.id}")

## 9. Stream: api_gateway_logs

In [ ]:
SOURCE_APIGW = "api_gateway_logs"

raw_apigw_stream = (
    spark.readStream
    .format("eventhubs")
    .options(**build_eventhubs_conf(eh_conn_apigw, consumer_group="cg-bronze-apigw"))
    .load()
)

enriched_apigw = enrich_stream(raw_apigw_stream, SOURCE_APIGW)

query_apigw = (
    enriched_apigw.writeStream
    .format("parquet")
    .option("checkpointLocation", checkpoint_path(SOURCE_APIGW))
    .option("path", bronze_path(SOURCE_APIGW))
    .partitionBy("ingestion_date", "source_system")
    .trigger(processingTime=f"{trigger_interval_s} seconds")
    .outputMode("append")
    .queryName(f"brz_{SOURCE_APIGW}")
    .start()
)

print(f"Streaming query '{query_apigw.name}' started — ID: {query_apigw.id}")

## 10. Monitor Streams Until Pipeline Termination

In [ ]:
import time

queries = [query_txn, query_auth, query_apigw]

def log_stream_progress(q):
    p = q.lastProgress
    if p:
        print(
            f"[{q.name}] batchId={p.get('batchId', 'N/A')} "
            f"inputRows={p.get('numInputRows', 0)} "
            f"processedRowsPerSec={p.get('processedRowsPerSecond', 0):.1f}"
        )

# In a Synapse pipeline the notebook runs for the pipeline's designated window.
# awaitTermination() blocks until the stream stops or an exception is raised.
try:
    for q in queries:
        q.awaitTermination(timeout=trigger_interval_s * 2)
        log_stream_progress(q)
    print("All streaming queries completed their trigger window.")
except Exception as exc:
    print(f"Stream error: {exc}")
    for q in queries:
        try:
            q.stop()
        except Exception:
            pass
    raise

## 11. Output Audit Summary

In [ ]:
summary = {}
for src, path in [
    (SOURCE_TXN,  bronze_path(SOURCE_TXN)),
    (SOURCE_AUTH, bronze_path(SOURCE_AUTH)),
    (SOURCE_APIGW,bronze_path(SOURCE_APIGW)),
]:
    try:
        count = spark.read.parquet(path).count()
    except Exception:
        count = -1   # path may not yet exist if no events arrived
    summary[src] = count
    print(f"  {src}: {count:,} rows in Bronze")

# Surface as notebook exit value for ADF monitoring
mssparkutils.notebook.exit(str(summary))